# Large Language Model Basics

**Following the tutorial** : https://www.youtube.com/watch?v=UU1WVnMk4E8

## 1. Loading the Dataset

In [3]:
with open("dataset/wizard_of_oz.txt", "r", encoding="utf-8") as f:
    text = f.read()
print(text[:200])

    DOROTHY AND THE WIZARD IN OZ

  BY

  L. FRANK BAUM

  AUTHOR OF THE WIZARD OF OZ, THE LAND OF OZ, OZMA OF OZ, ETC.

  ILLUSTRATED BY JOHN R. NEILL

  BOOKS OF WONDER WILLIAM MORROW & CO., INC. NE


In [5]:
chars = sorted(set(text))
vocabulary_len = len(chars)
print(chars)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [8]:
encode_chars = {c:i for i,c in enumerate(chars)}
decode_chars = {i:c for i,c in enumerate(chars)}

def encode(text):
    return [encode_chars.get(i) for i in text]

def decode(char_list):
    return ''.join([decode_chars.get(i) for i in char_list]) 
    

In [12]:
enc = encode("hello world")
print(f"Encoding: {enc} , \nDecoding: {decode(enc)}")

Encoding: [61, 58, 65, 65, 68, 1, 76, 68, 71, 65, 57] , 
Decoding: hello world


## 2. pytorch nn

In [13]:
import torch

In [14]:
data = torch.tensor(encode(text), dtype=torch.long)
data[:100]

tensor([ 1,  1,  1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32,
        29,  1, 47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1,
        26, 49,  0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45,
        37,  0,  0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,
         1, 47, 33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,
         1, 36, 25, 38, 28,  1, 39, 30,  1, 39])

In [23]:
n = int(0.8*len(data))
train = data[:n]
test = data[n:]
print(f"Train len :{len(train)} \nTest len: {len(test)}")

n = int(0.8*len(train))
val = train[n:]
train = train[:n]
print(f"Train len :{len(train)} \nVal len: {len(val)}")

Train len :185984 
Test len: 46496
Train len :148787 
Val len: 37197


In [19]:
block_size = 8
x = train[:block_size]
y = train[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When input is {context} target is {target}")

When input is tensor([1]) target is 1
When input is tensor([1, 1]) target is 1
When input is tensor([1, 1, 1]) target is 1
When input is tensor([1, 1, 1, 1]) target is 28
When input is tensor([ 1,  1,  1,  1, 28]) target is 39
When input is tensor([ 1,  1,  1,  1, 28, 39]) target is 42
When input is tensor([ 1,  1,  1,  1, 28, 39, 42]) target is 39
When input is tensor([ 1,  1,  1,  1, 28, 39, 42, 39]) target is 44


In [31]:
batch_size = 4
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"device : {device}")

def get_batch(split):
    d = train if split == "train" else val
    ix = torch.randint(len(d) - block_size, (batch_size,))
    #print(ix)
    x = torch.stack([d[i:(i+block_size)] for i in ix])
    y = torch.stack([d[i+1:(i+block_size) + 1] for i in ix])

    x,y = x.to(device=device), y.to(device=device)
    return x,y

x,y = get_batch("train")
print(f"input : {x} \ntarget : {y}")

device : cpu
input : tensor([[73, 58, 67, 73, 58, 57,  1, 62],
        [72, 65, 58, 58, 69,  1, 78, 58],
        [ 1, 73, 61, 58,  1, 56, 68, 66],
        [61, 62, 67, 64,  1, 72, 68, 11]]) 
target : tensor([[58, 67, 73, 58, 57,  1, 62, 67],
        [65, 58, 58, 69,  1, 78, 58, 73],
        [73, 61, 58,  1, 56, 68, 66, 55],
        [62, 67, 64,  1, 72, 68, 11,  1]])


In [32]:
import torch.nn as nn
import torch.nn.functional as F

#### Documentation
 
a. nn.Embedding => https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html

b. nn.CrossEntropyLoss => https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html#torch.nn.CrossEntropyLoss

c. nn.Layer => https://pytorch.org/docs/stable/generated/torch.nn.Linear.html#torch.nn.Linear

In [94]:
# exploring embedding , it has reandomly init weights
emb = nn.Embedding(num_embeddings=10, embedding_dim=3)

context = torch.tensor([1])
print(f"Context {context}, embedding  for the context: {emb(context)}")

context = torch.tensor([2])
print(f"Context {context}, embedding  for the context: {emb(context)}")

try:
    context = torch.tensor([10])
    print(f"Context {context}, embedding  for the context: {emb(context)}")
except:
    print("Fail")

Context tensor([1]), embedding  for the context: tensor([[ 0.5720, -1.8768, -1.0206]], grad_fn=<EmbeddingBackward0>)
Context tensor([2]), embedding  for the context: tensor([[ 0.4978,  0.2826, -0.8286]], grad_fn=<EmbeddingBackward0>)
Fail


In [92]:
# if the index tensor is has inc dim, results reflects it as well
context = torch.tensor([1])
ex = emb(context)
print(f"Context {context}, embedding {ex}, shape {ex.shape}")

context = torch.tensor([[1]])
ex = emb(context)
print(f"Context {context}, embedding {ex}, shape {ex.shape}")

Context tensor([1]), embedding tensor([[-0.1409,  1.3608,  0.8399]], grad_fn=<EmbeddingBackward0>), shape torch.Size([1, 3])
Context tensor([[1]]), embedding tensor([[[-0.1409,  1.3608,  0.8399]]], grad_fn=<EmbeddingBackward0>), shape torch.Size([1, 1, 3])


In [95]:
# if using pretrained weights
weight = torch.FloatTensor([[1, 2.3, 3], [4, 5.1, 6.3]])
embedding = nn.Embedding.from_pretrained(weight)
input = torch.LongTensor([1])
embedding(input)

tensor([[4.0000, 5.1000, 6.3000]])

In [108]:
# view
a = torch.randn(4, 2)
print(f"a : {a},\nwith view {a.view(8)}")
print(f"a : {a},\nwith view {a.view((2,4))}")

a : tensor([[ 0.1444,  0.5487],
        [ 1.5782,  2.4542],
        [-1.3371, -0.8412],
        [ 0.8863,  0.1287]]),
with view tensor([ 0.1444,  0.5487,  1.5782,  2.4542, -1.3371, -0.8412,  0.8863,  0.1287])
a : tensor([[ 0.1444,  0.5487],
        [ 1.5782,  2.4542],
        [-1.3371, -0.8412],
        [ 0.8863,  0.1287]]),
with view tensor([[ 0.1444,  0.5487,  1.5782,  2.4542],
        [-1.3371, -0.8412,  0.8863,  0.1287]])


In [113]:
# eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250

In [109]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # same as faltening it, eg: emb of shape (2, 4, 60), will be changed to (8, 60)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)  # cross entroy requires batch size first then the number of classes
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size=vocabulary_len)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)



UqEfO:HJtIr]dtt?A
PkEIRg5h"ucaJesJFNFn-Q(tuj*U_ 5CML3nWcUSWG,IN"w9jrnPK18vKz:sWpPRHaxAc[CMJ6kPZhEOp&m(M(ACsJbFNOL&)c7?8GH0;4IuX6nHc
qV2&C*k&nj1sJ91ZnwMBFVUCDNj*;GLLHdIa&"r;YL&:6ZrWt.'EHf4u5UWHQkK3NH(?d)4ujYVOS[)ly3&lPk.pdIu[Mc;xLmmL3g!bXqm!?1]FVB*IP67u0otu 4,ZiVOOS7r[coUz!8UD3&SK6lYMgP uyBXFd&([i3Sd_3tufJbdh]f;uvN
vO)LHPT""u[F wIn__jRFX7YkKnA]Rm?xwYH!NcUs09cBaE*H!1bwrT"EHzw;t*nkN1;KzX6N80K6W5-0!L'c
cUaKgKu_0?
l.FVpmiIAUNeNm&Rl)4Lp3yWH7jpJ?K6aS1Z7sE,K6N4P
f7z,Fq-yRjPGJ-G_mJY.q4v5CUV5)wz6_OJ4!V g


In [111]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [118]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

max_iters = 50000

for iter in range(max_iters):
    if iter % 500 == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 2.553, val loss: 2.581
step: 250, train loss: 2.535, val loss: 2.564
step: 500, train loss: 2.523, val loss: 2.567
step: 750, train loss: 2.518, val loss: 2.576
step: 1000, train loss: 2.533, val loss: 2.585
step: 1250, train loss: 2.523, val loss: 2.569
step: 1500, train loss: 2.527, val loss: 2.560
step: 1750, train loss: 2.525, val loss: 2.553
step: 2000, train loss: 2.528, val loss: 2.536
step: 2250, train loss: 2.533, val loss: 2.584
step: 2500, train loss: 2.529, val loss: 2.557
step: 2750, train loss: 2.535, val loss: 2.528
step: 3000, train loss: 2.517, val loss: 2.552
step: 3250, train loss: 2.495, val loss: 2.558
step: 3500, train loss: 2.534, val loss: 2.540
step: 3750, train loss: 2.504, val loss: 2.579
step: 4000, train loss: 2.499, val loss: 2.547
step: 4250, train loss: 2.498, val loss: 2.529
step: 4500, train loss: 2.531, val loss: 2.530
step: 4750, train loss: 2.506, val loss: 2.540
step: 5000, train loss: 2.505, val loss: 2.547
step: 5250, train l

In [119]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=100)[0].tolist())
print(generated_chars)


"BL. thed y. okalithal Sofr f s he EAxed ngse t shint thorlirid me o ans fat thice tifa hath, be clo


In [147]:
enc = encode("hor")
decode_ls = decode(m.generate(torch.tensor([enc]), max_new_tokens=5)[0].tolist())
print(f"Next 5 tokens : {decode_ls}")

enc = encode("The boy")
decode_ls = decode(m.generate(torch.tensor([enc]), max_new_tokens=10)[0].tolist())
print(f"Next 10 tokens : {decode_ls}")

Next 5 tokens : horthani
Next 10 tokens : The boy out asess
